In [1]:
import sqlite3
import requests
import pandas as pd

ES_URL = 'https://leasing-prod.es.us-east-1.aws.found.io/listings-v2/_search'

headers = {
    'accept': 'application/json',
    'authorization': 'apiKey Z3JqdWNKRUJCaElILXdxR1lNS2Q6Z3hTc0RQQkVUSVNOcWZ6QVRUOTVHQQ==',
    'content-type': 'application/json',
    'origin': 'https://www.msrenewal.com',
    'referer': 'https://www.msrenewal.com/',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36',
}

In [2]:
market_name = input('Enter market name (e.g. Houston, Atlanta, Dallas, Phoenix, Tampa): ').strip()
state_code = input('Enter state code (e.g. TX, GA, AZ, FL): ').strip().upper()

metro_location = f'{market_name.lower()}-{state_code.lower()}'
print(f'Searching: {market_name}, {state_code}')

Searching: houston, TX


In [3]:
query = {
    'from': 0,
    'size': 1500,
    'query': {
        'bool': {
            'must': [
                {'match': {'Record_Type_Text__c': 'Leasing'}},
                {'term': {'Syndicate_MSR__c': {'value': True}}},
                {'bool': {
                    'should': [
                        {'match': {'Listing_Status__c': 'Active'}},
                        {'match': {'Listing_Status__c': 'Coming Soon'}},
                        {'match': {'Listing_Status__c': 'In-Repair'}},
                    ],
                    'must_not': {'term': {'Hide_Listing_for_Pending_Execution__c': {'value': True}}},
                }},
                {'match': {'Market__r.Name': market_name}},
                {'match': {'Market__r.State__c': state_code}},
            ]
        }
    },
    '_source': [
        'Name', 'Listing_Status__c', 'Premium_Listing__c', 'Hot_Home__c',
        'Market__r.Name', 'Market__r.State__c',
        'Property__r.Name', 'Property__r.City__c', 'Property__r.State_Code__c',
        'Property__r.Zipcode__c', 'Property__r.Beds__c', 'Property__r.Baths__c',
        'Property__r.Square_Ft__c', 'Property__r.Rent__c', 'Property__r.Year_Built__c',
        'Property__r.Available_Date__c', 'Property__r.Latitude__c', 'Property__r.Longitude__c',
        'Property__r.Zillow_3d_Link__c',
        'Property__r.Specials__r.Marketing_Description__c',
    ],
}

response = requests.post(ES_URL, json=query, headers=headers)
response.raise_for_status()

data = response.json()
hits = data['hits']['hits']
total = data['hits']['total']['value']

records = []
for hit in hits:

    s = hit['_source']
    prop = s.get('Property__r', {})
    specials = prop.get('Specials__r', {})
    if isinstance(specials, list) and specials:
        special_text = specials[0].get('Marketing_Description__c', '')
    elif isinstance(specials, dict):
        special_text = specials.get('Marketing_Description__c', '')
    else:
        special_text = ''

    records.append({
        'name': s.get('Name', ''),

        'street': prop.get('Name', ''),
        'city': prop.get('City__c', ''),
        'state': prop.get('State_Code__c', ''),
        'zip': prop.get('Zipcode__c', ''),
        'beds': prop.get('Beds__c', ''),
        'baths': prop.get('Baths__c', ''),
        'sqft': prop.get('Square_Ft__c', ''),
        'rent': prop.get('Rent__c', ''),
        'year_built': prop.get('Year_Built__c', ''),
        'available_date': prop.get('Available_Date__c', ''),
        'listing_status': s.get('Listing_Status__c', ''),
        'premium': s.get('Premium_Listing__c', False),
        'hot_home': s.get('Hot_Home__c', False),
        'special': special_text,
        'lat': prop.get('Latitude__c', ''),
        'lng': prop.get('Longitude__c', ''),
        'virtual_tour': prop.get('Zillow_3d_Link__c', ''),
        'metro_location': metro_location,
        'link': f'https://www.msrenewal.com/home/{s.get('Name','').replace(' ','-').replace(',','')}/{hit.get("_id")}',
    })

print(f'Total: {total} listings found, {len(records)} scraped')

Total: 61 listings found, 61 scraped


In [4]:
data

{'took': 7,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 61, 'relation': 'eq'},
  'max_score': 11.068949,
  'hits': [{'_index': 'listings-v2-2026.02.03-17.26.29.147z',
    '_id': 'a1p4u000003wE7pAAE',
    '_score': 11.068949,
    '_source': {'Listing_Status__c': 'In-Repair',
     'Premium_Listing__c': False,
     'Hot_Home__c': False,
     'Property__r': {'Zillow_3d_Link__c': None,
      'Square_Ft__c': 1900,
      'Year_Built__c': '1979',
      'Baths__c': 2,
      'Name': '1434 New Tree Ln',
      'City__c': 'Missouri City',
      'Rent__c': 2065,
      'Latitude__c': 29.5929508209229,
      'Longitude__c': -95.52287292480469,
      'Available_Date__c': '2026-03-06',
      'Beds__c': 4,
      'State_Code__c': 'TX',
      'Zipcode__c': '77489'},
     'Name': '1434 New Tree Ln, Missouri City, TX 77489',
     'Market__r': {'State__c': 'TX', 'Name': 'Houston'}}},
   {'_index': 'listings-v2-2026.02.03-17.26.29.147z

In [5]:
df = pd.DataFrame(records)

dupes = df.duplicated(subset='name', keep='first').sum()
print(f'Duplicate rows: {dupes}')
df = df.drop_duplicates(subset='name', keep='first')

df.info()
df.head()

Duplicate rows: 0
<class 'pandas.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            61 non-null     str    
 1   street          61 non-null     str    
 2   city            61 non-null     str    
 3   state           61 non-null     str    
 4   zip             61 non-null     str    
 5   beds            61 non-null     int64  
 6   baths           61 non-null     float64
 7   sqft            61 non-null     int64  
 8   rent            61 non-null     int64  
 9   year_built      61 non-null     str    
 10  available_date  61 non-null     str    
 11  listing_status  61 non-null     str    
 12  premium         61 non-null     bool   
 13  hot_home        61 non-null     bool   
 14  special         61 non-null     str    
 15  lat             61 non-null     float64
 16  lng             61 non-null     float64
 17  virtual_tour    6 non-null    

,name,street,city,state,zip,beds,baths,sqft,rent,year_built,available_date,listing_status,premium,hot_home,special,lat,lng,virtual_tour,metro_location,link
0,"1434 New Tree Ln, Missouri City, TX 77489",1434 New Tree Ln,Missouri City,TX,77489,4,2.0,1900,2065,1979,2026-03-06,In-Repair,False,False,,29.592951,-95.522873,NaN,houston-tx,https://www.msrenewal.com/home/1434-New-Tree-L...
1,"12418 Meadow Briar Dr, Meadows, TX 77477",12418 Meadow Briar Dr,Stafford,TX,77477,4,2.0,1900,2190,1985,2026-02-23,In-Repair,False,False,,29.640329,-95.593468,NaN,houston-tx,https://www.msrenewal.com/home/12418-Meadow-Br...
2,"2127 ROUND WIND TRL, CROSBY, TX 77532",2127 ROUND WIND TRL,CROSBY,TX,77532,4,3.0,2507,2280,1978,2026-03-07,In-Repair,False,False,,29.977658,-95.118901,NaN,houston-tx,https://www.msrenewal.com/home/2127-ROUND-WIND...
3,"14906 Loys Coves Ct, Humble, TX 77396",14906 Loys Coves Ct,Humble,TX,77396,4,3.0,2103,2115,2005,2026-03-27,Coming Soon,False,False,,29.938250,-95.216090,NaN,houston-tx,https://www.msrenewal.com/home/14906-Loys-Cove...
4,"7003 Oakmantle Dr, Houston, TX 77085",7003 Oakmantle Dr,Houston,TX,77085,4,2.0,1820,1980,1985,2026-03-21,Coming Soon,False,False,,29.627950,-95.490280,NaN,houston-tx,https://www.msrenewal.com/home/7003-Oakmantle-...


In [6]:
df.to_csv('main_street_renewal.csv', index=False)
print(f'Saved {len(df)} rows to main_street_renewal.csv')

Saved 61 rows to main_street_renewal.csv


In [7]:
conn = sqlite3.connect('main_street_renewal.db')
df.to_sql('listings', conn, if_exists='replace', index=False)
print(f'Wrote {len(df)} rows to main_street_renewal.db -> listings table')
conn.close()

Wrote 61 rows to main_street_renewal.db -> listings table
